# 🏷️ Phase 2: Xây Dựng Bộ Nhãn Yếu (Weak Labels) cho Verifier

**Mục tiêu:**  
Từ predictions thu thập ở Phase 1, tạo nhãn binary `{good_rationale=1, bad_rationale=0}`  
dựa trên ground-truth answer — **không cần gán nhãn mới**.

**Chiến lược gán nhãn yếu:**
- `A_pred == A_gt` → Rationale đúng hướng → `weak_label = 1`
- `A_pred != A_gt` → Rationale có khả năng sai → `weak_label = 0`

**Hạn chế đã biết (cần đề cập trong thesis):**
- *Lucky guess*: A đúng ngẫu nhiên dù R sai → False Positive  
- *Correct reasoning, wrong answer*: R đúng nhưng model không decode được → False Negative  
→ Chính vì thế Phase 3 cần CLIP verification để bổ sung tín hiệu độc lập

**Output:**  
- `verifier_data/{dataset}_{train/val/test}.csv` (stratified split 60/20/20)
- Thống kê imbalance, per-region breakdown
- Plots phân bố

**⏱ Thời gian:** ~30 phút (CPU only, không cần GPU)

---
> **Prerequisite:** Đã chạy xong `01_reproduce_baseline.ipynb` và có files  
> `predictions_{dataset}_reasoning.csv` trong `OUTPUT_DIR`


## ⚙️ Cell 0: Mount Drive & Cấu Hình Đường Dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ============================================================
# ✏️ CHỈNH SỬA NẾU CẦN (phải khớp với Phase 1)
# ============================================================
PROJECT_ROOT   = '/content/drive/MyDrive/MedThink_Verifier'
PHASE1_OUTPUT  = f'{PROJECT_ROOT}/phase1_predictions'
VERIFIER_DIR   = f'{PROJECT_ROOT}/verifier_data'
PLOTS_DIR      = f'{PROJECT_ROOT}/phase2_plots'

os.makedirs(VERIFIER_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

DATASET_NAMES  = ['R-RAD', 'R-SLAKE', 'R-Path']
STRATEGIES     = ['reasoning', 'explanation']

print(f'PROJECT_ROOT  = {PROJECT_ROOT}')
print(f'PHASE1_OUTPUT = {PHASE1_OUTPUT}')
print(f'VERIFIER_DIR  = {VERIFIER_DIR}')

# Kiểm tra files Phase 1 có sẵn
print('\n📂 Checking Phase 1 output files:')
missing_files = []
for ds in DATASET_NAMES:
    for st in STRATEGIES:
        fpath = f'{PHASE1_OUTPUT}/predictions_{ds}_{st}.csv'
        exists = os.path.exists(fpath)
        size = os.path.getsize(fpath) / 1024 if exists else 0
        status = f'✅ ({size:.0f} KB)' if exists else '❌ MISSING'
        print(f'  {status}  predictions_{ds}_{st}.csv')
        if not exists:
            missing_files.append(fpath)

if missing_files:
    print(f'\n⚠️ {len(missing_files)} file(s) missing. Hãy chạy Phase 1 trước!')

## 📦 Cell 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

print('Libraries imported ✅')

## 🏷️ Cell 2: Hàm Tạo Weak Labels

In [ ]:
def create_weak_labels(csv_path, verbose=True):
    """
    Đọc predictions CSV và gán nhãn yếu.
    
    Nhãn yếu:
    - weak_label = 1: A_pred == A_gt → rationale likely good
    - weak_label = 0: A_pred != A_gt → rationale likely bad
    
    Trả về: DataFrame đã thêm cột 'weak_label' và metadata
    """
    if not os.path.exists(csv_path):
        print(f'❌ File not found: {csv_path}')
        return None
    
    df = pd.read_csv(csv_path)
    
    # Weak label từ is_correct
    if 'is_correct' in df.columns:
        df['weak_label'] = df['is_correct'].astype(int)
    elif 'answer_pred' in df.columns and 'answer_gt' in df.columns:
        # Recompute nếu is_correct không có
        df['is_correct'] = (
            df['answer_pred'].str.strip().str.lower() == 
            df['answer_gt'].str.strip().str.lower()
        )
        df['weak_label'] = df['is_correct'].astype(int)
    else:
        print(f'❌ Không tìm thấy cột is_correct hoặc answer_pred/answer_gt')
        return None
    
    # Metadata
    n_total = len(df)
    n_good  = df['weak_label'].sum()
    n_bad   = n_total - n_good
    pos_rate = n_good / n_total if n_total > 0 else 0
    imbalance_ratio = n_good / n_bad if n_bad > 0 else float('inf')
    
    # Rationale length statistics
    if 'rationale_pred' in df.columns:
        df['rationale_len_words'] = df['rationale_pred'].apply(
            lambda x: len(str(x).split()) if pd.notna(x) and str(x).strip() != '' else 0
        )
        df['rationale_len_chars'] = df['rationale_pred'].apply(
            lambda x: len(str(x)) if pd.notna(x) else 0
        )
        df['rationale_is_empty'] = df['rationale_len_words'] < 5
    
    if verbose:
        print(f'  Total:   {n_total}')
        print(f'  Good (1): {n_good} ({pos_rate:.1%})')
        print(f'  Bad  (0): {n_bad} ({1-pos_rate:.1%})')
        print(f'  Imbalance ratio (good/bad): {imbalance_ratio:.2f}')
        
        if 'rationale_len_words' in df.columns:
            empty_count = df['rationale_is_empty'].sum()
            print(f'  Empty/too-short rationales (<5w): {empty_count} ({empty_count/n_total:.1%})')
            print(f'  Avg rationale len — Good: {df[df["weak_label"]==1]["rationale_len_words"].mean():.1f}w')
            print(f'  Avg rationale len — Bad:  {df[df["weak_label"]==0]["rationale_len_words"].mean():.1f}w')
    
    return df


print('create_weak_labels() defined ✅')

## ✂️ Cell 3: Hàm Chia Train/Val/Test Stratified

In [ ]:
def split_verifier_data(df, dataset_name, strategy, 
                        train_ratio=0.60, val_ratio=0.20, test_ratio=0.20,
                        random_state=42):
    """
    Chia dataset cho training verifier với stratification.
    
    Splits:
    - train: 60% — train verifier (Logistic Regression / RF)
    - val:   20% — chọn threshold adaptive (Adaptive F1)
    - test:  20% — đánh giá final, báo cáo
    
    Stratified theo weak_label để đảm bảo cân bằng class
    trong mỗi split.
    
    Returns:
        (train_df, val_df, test_df)
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9, \
        'Tỷ lệ train/val/test phải cộng bằng 1.0'
    
    # Kiểm tra đủ mẫu cho stratification
    n_classes = df['weak_label'].nunique()
    if n_classes < 2:
        print(f'⚠️ {dataset_name}/{strategy}: Chỉ có 1 class — không thể stratify')
        # Fallback: random split
        train_df = df.sample(frac=train_ratio, random_state=random_state)
        remaining = df.drop(train_df.index)
        val_df = remaining.sample(frac=val_ratio/(val_ratio+test_ratio), random_state=random_state)
        test_df = remaining.drop(val_df.index)
    else:
        # Stratified split
        test_size = val_ratio + test_ratio
        train_df, temp_df = train_test_split(
            df, test_size=test_size, 
            stratify=df['weak_label'], 
            random_state=random_state
        )
        
        val_fraction = val_ratio / test_size
        val_df, test_df = train_test_split(
            temp_df, test_size=1.0 - val_fraction,
            stratify=temp_df['weak_label'],
            random_state=random_state
        )
    
    # Thêm cột split
    train_df = train_df.copy(); train_df['split'] = 'train'
    val_df   = val_df.copy();   val_df['split']   = 'val'
    test_df  = test_df.copy();  test_df['split']  = 'test'
    
    # Lưu files
    prefix = f'{VERIFIER_DIR}/{dataset_name}_{strategy}'
    train_df.to_csv(f'{prefix}_train.csv', index=False)
    val_df.to_csv(f'{prefix}_val.csv',   index=False)
    test_df.to_csv(f'{prefix}_test.csv',  index=False)
    
    # Print summary
    print(f'  Train: {len(train_df):4d} (pos rate: {train_df["weak_label"].mean():.1%})')
    print(f'  Val:   {len(val_df):4d} (pos rate: {val_df["weak_label"].mean():.1%})')
    print(f'  Test:  {len(test_df):4d} (pos rate: {test_df["weak_label"].mean():.1%})')
    
    return train_df, val_df, test_df


print('split_verifier_data() defined ✅')

## ▶️ Cell 4: Chạy Toàn Bộ — Tạo Weak Labels & Chia Split

In [ ]:
all_labeled_dfs  = {}  # key: (dataset, strategy)
all_split_dfs    = {}  # key: (dataset, strategy, split)

GLOBAL_SUMMARY   = []

for ds_name in DATASET_NAMES:
    for strategy in STRATEGIES:
        csv_path = f'{PHASE1_OUTPUT}/predictions_{ds_name}_{strategy}.csv'
        
        print(f'\n{"─"*60}')
        print(f'🏷️  {ds_name} | strategy={strategy}')
        print(f'{"─"*60}')
        
        # Tạo weak labels
        df = create_weak_labels(csv_path, verbose=True)
        if df is None:
            print(f'  ⏭️ Skipping (file not found or error)')
            continue
        
        all_labeled_dfs[(ds_name, strategy)] = df
        
        # Chia split
        print('\n  Splitting 60/20/20 (stratified):')
        train_df, val_df, test_df = split_verifier_data(df, ds_name, strategy)
        all_split_dfs[(ds_name, strategy, 'train')] = train_df
        all_split_dfs[(ds_name, strategy, 'val')]   = val_df
        all_split_dfs[(ds_name, strategy, 'test')]  = test_df
        
        # Ghi vào summary
        n_total = len(df)
        n_good  = df['weak_label'].sum()
        GLOBAL_SUMMARY.append({
            'Dataset':     ds_name,
            'Strategy':    strategy,
            'N_total':     n_total,
            'N_good (1)':  int(n_good),
            'N_bad (0)':   int(n_total - n_good),
            'Pos_rate':    round(n_good / n_total * 100, 1),
            'N_train':     len(train_df),
            'N_val':       len(val_df),
            'N_test':      len(test_df),
        })

print(f'\n{"="*60}')
print('ALL DONE — Summary:')
print('='*60)
summary_df = pd.DataFrame(GLOBAL_SUMMARY)
print(summary_df.to_string(index=False))

summary_df.to_csv(f'{VERIFIER_DIR}/weak_label_summary.csv', index=False)
print(f'\n💾 Saved → {VERIFIER_DIR}/weak_label_summary.csv')

## 📊 Cell 5: Phân Tích Class Imbalance

In [ ]:
n_datasets = len(DATASET_NAMES)
n_strategies = len(STRATEGIES)

fig, axes = plt.subplots(
    n_strategies, n_datasets, 
    figsize=(5 * n_datasets, 4 * n_strategies),
    squeeze=False
)
fig.suptitle('Class Imbalance: Good vs Bad Rationales (Weak Labels)', 
             fontsize=14, fontweight='bold', y=1.02)

colors_bar = ['#4CAF50', '#F44336']  # green=good, red=bad

for row_idx, strategy in enumerate(STRATEGIES):
    for col_idx, ds_name in enumerate(DATASET_NAMES):
        ax = axes[row_idx][col_idx]
        key = (ds_name, strategy)
        
        if key not in all_labeled_dfs:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{ds_name}\n({strategy})', fontsize=11)
            continue
        
        df = all_labeled_dfs[key]
        counts = df['weak_label'].value_counts().sort_index()
        labels_map = {0: 'Bad (0)', 1: 'Good (1)'}
        
        bars = ax.bar(
            [labels_map.get(k, str(k)) for k in counts.index],
            counts.values,
            color=[colors_bar[k] for k in counts.index],
            edgecolor='white', linewidth=1.2, alpha=0.85
        )
        
        # Annotate counts + percentages
        total = len(df)
        for bar, val in zip(bars, counts.values):
            pct = val / total * 100
            ax.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(counts.values) * 0.01,
                f'{val}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=10
            )
        
        title_strategy = 'Reasoning (R→A)' if strategy == 'reasoning' else 'Explanation (A→R)'
        ax.set_title(f'{ds_name}\n{title_strategy}\nN={total}', fontsize=11)
        ax.set_xlabel('Weak Label')
        ax.set_ylabel('Count')
        ax.set_ylim(0, max(counts.values) * 1.2)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/imbalance_weak_labels.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'💾 Saved → {PLOTS_DIR}/imbalance_weak_labels.png')

## 📈 Cell 6: So Sánh Phân Phối Độ Dài Rationale — Good vs Bad

In [ ]:
# Kiểm tra: Bad rationale có ngắn hơn Good rationale không?
# (tín hiệu ban đầu về chất lượng rationale)

fig, axes = plt.subplots(
    n_strategies, n_datasets,
    figsize=(5 * n_datasets, 4 * n_strategies),
    squeeze=False
)
fig.suptitle('Rationale Length Distribution: Good vs Bad (Weak Labels)',
             fontsize=14, fontweight='bold', y=1.02)

for row_idx, strategy in enumerate(STRATEGIES):
    for col_idx, ds_name in enumerate(DATASET_NAMES):
        ax = axes[row_idx][col_idx]
        key = (ds_name, strategy)
        
        if key not in all_labeled_dfs:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue
        
        df = all_labeled_dfs[key]
        if 'rationale_len_words' not in df.columns:
            ax.text(0.5, 0.5, 'No rationale length data', ha='center', va='center',
                    transform=ax.transAxes)
            continue
        
        good = df[df['weak_label'] == 1]['rationale_len_words']
        bad  = df[df['weak_label'] == 0]['rationale_len_words']
        
        bins = np.linspace(0, df['rationale_len_words'].quantile(0.99), 40)
        ax.hist(good, bins=bins, alpha=0.6, color='#4CAF50', label=f'Good (μ={good.mean():.0f}w)', density=True)
        ax.hist(bad,  bins=bins, alpha=0.6, color='#F44336', label=f'Bad  (μ={bad.mean():.0f}w)',  density=True)
        
        # Mean lines
        ax.axvline(good.mean(), color='#2E7D32', linestyle='--', linewidth=1.5)
        ax.axvline(bad.mean(),  color='#C62828', linestyle='--', linewidth=1.5)
        
        title_strategy = 'Reasoning' if strategy == 'reasoning' else 'Explanation'
        ax.set_title(f'{ds_name} — {title_strategy}', fontsize=11)
        ax.set_xlabel('Rationale Length (words)')
        ax.set_ylabel('Density')
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/rationale_len_good_vs_bad.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'💾 Saved → {PLOTS_DIR}/rationale_len_good_vs_bad.png')

## 🧭 Cell 7: Phân Tích Theo Anatomical Region (R-RAD)

Xác nhận pattern từ Bảng 4: Pelvic Cavity có error rate cao nhất → imbalance mạnh.

In [ ]:
# Chỉ phân tích R-RAD Reasoning (nơi quan trọng nhất)
key_rrad = ('R-RAD', 'reasoning')

if key_rrad in all_labeled_dfs:
    df_rrad = all_labeled_dfs[key_rrad].copy()
    
    # Tìm anatomical region field
    region_cols = [c for c in df_rrad.columns 
                   if any(k in c.lower() for k in ['region', 'organ', 'body', 'anatomy', 'category'])]
    
    print(f'Available columns in R-RAD: {list(df_rrad.columns)}')
    print(f'Potential region columns: {region_cols}')
    
    if region_cols:
        region_col = region_cols[0]
        print(f'→ Using column: {region_col}')
        
        region_stats = df_rrad.groupby(region_col).agg(
            N                = ('weak_label', 'count'),
            N_good_label_1   = ('weak_label', 'sum'),
            Accuracy_pct     = ('is_correct',  lambda x: round(x.mean()*100, 1)),
            Pos_rate_pct     = ('weak_label',  lambda x: round(x.mean()*100, 1)),
        ).reset_index()
        region_stats['N_bad_label_0'] = region_stats['N'] - region_stats['N_good_label_1']
        region_stats = region_stats.sort_values('Accuracy_pct')
        
        print('\n📊 R-RAD per-Region Analysis:')
        print(region_stats.to_string(index=False))
        
        # Flag regions với N < 30 → cần cross-dataset threshold
        small_n_regions = region_stats[region_stats['N'] < 30][region_col].tolist()
        if small_n_regions:
            print(f'\n⚠️ Regions với N<30 (cần cross-dataset threshold): {small_n_regions}')
        
        # Visualize
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Accuracy per region
        colors_region = ['#EF5350' if n < 30 else '#42A5F5' 
                         for n in region_stats['N']]
        axes[0].barh(region_stats[region_col], region_stats['Accuracy_pct'],
                     color=colors_region, alpha=0.85, edgecolor='white')
        axes[0].axvline(x=region_stats['Accuracy_pct'].mean(), 
                        color='black', linestyle='--', label='Mean accuracy')
        axes[0].set_xlabel('Accuracy (%)')
        axes[0].set_title('Accuracy per Anatomical Region\n(red = N<30)', fontsize=12)
        axes[0].legend()
        
        # N per region
        axes[1].barh(region_stats[region_col], region_stats['N'],
                     color='#7E57C2', alpha=0.85, edgecolor='white')
        axes[1].axvline(x=30, color='red', linestyle='--', label='N=30 threshold')
        axes[1].set_xlabel('Sample Count')
        axes[1].set_title('Sample Size per Region', fontsize=12)
        axes[1].legend()
        
        plt.tight_layout()
        plt.savefig(f'{PLOTS_DIR}/rrad_per_region_analysis.png', bbox_inches='tight', dpi=150)
        plt.show()
        
        region_stats.to_csv(f'{VERIFIER_DIR}/rrad_per_region_analysis.csv', index=False)
        print(f'💾 Saved → {VERIFIER_DIR}/rrad_per_region_analysis.csv')
    
    else:
        print('⚠️ Không tìm thấy cột anatomical region.')
        print('→ Cần thêm metadata từ VQA-RAD gốc (xem Cell 7B).')
else:
    print('⚠️ Không có data R-RAD/reasoning')

## 🔗 Cell 7B: Join Anatomical Region Từ VQA-RAD Metadata

> Chỉ cần chạy nếu Cell 7 báo thiếu cột anatomical region

In [ ]:
# VQA-RAD metadata — download nếu cần
# Source: https://osf.io/89kps/ (VQA-RAD dataset)

VQARAD_META_PATH = f'{PROJECT_ROOT}/Medthink_repo/Medthink_Dataset/R-RAD/test.json'

if os.path.exists(VQARAD_META_PATH):
    import json
    with open(VQARAD_META_PATH) as f:
        meta = json.load(f)
    
    print(f'Loaded {len(meta)} VQA-RAD records')
    print(f'Fields: {list(meta[0].keys()) if meta else "empty"}')
    
    # Build mapping: image_id → anatomy
    id2anatomy = {}
    for item in meta:
        img_id = item.get('image_id', item.get('img_id', item.get('image_name', '')))
        anatomy = item.get('image_organ', item.get('body_part', item.get('q_body_part', 'Unknown')))
        id2anatomy[str(img_id)] = anatomy
    
    print(f'Unique anatomical regions: {set(id2anatomy.values())}')
    
    # Thêm vào R-RAD dataframes
    for strategy in STRATEGIES:
        key = ('R-RAD', strategy)
        if key in all_labeled_dfs:
            df = all_labeled_dfs[key]
            df['anatomical_region'] = df['image_id'].astype(str).map(id2anatomy).fillna('Unknown')
            all_labeled_dfs[key] = df
            print(f'✅ Added anatomical_region to R-RAD/{strategy}')
            print(f'   Distribution: {dict(df["anatomical_region"].value_counts())}')

else:
    print(f'⚠️ VQA-RAD metadata not found at: {VQARAD_META_PATH}')
    print('→ Cần tải VQA-RAD gốc từ: https://osf.io/89kps/')

## 🎯 Cell 8: Phân Tích Answer Type — Closed vs Open

In [ ]:
# Phân tích accuracy và weak label distribution theo loại câu hỏi
fig, axes = plt.subplots(1, n_datasets, figsize=(6 * n_datasets, 5))
if n_datasets == 1:
    axes = [axes]

fig.suptitle('Error Rate by Answer Type (Reasoning Strategy)',
             fontsize=13, fontweight='bold')

for col_idx, ds_name in enumerate(DATASET_NAMES):
    ax = axes[col_idx]
    key = (ds_name, 'reasoning')
    
    if key not in all_labeled_dfs:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(ds_name)
        continue
    
    df = all_labeled_dfs[key]
    
    # Detect answer type field
    ans_type_col = None
    for c in ['answer_type', 'ans_type', 'closed_ended', 'question_type']:
        if c in df.columns:
            ans_type_col = c
            break
    
    if ans_type_col is None:
        # Infer từ answer_gt: Yes/No/0/1 → Closed, else Open
        closed_answers = {'yes', 'no', '0', '1', 'true', 'false'}
        df['answer_type_inferred'] = df['answer_gt'].apply(
            lambda x: 'Closed' if str(x).strip().lower() in closed_answers else 'Open'
        )
        ans_type_col = 'answer_type_inferred'
    
    type_stats = df.groupby(ans_type_col).agg(
        N            = ('weak_label', 'count'),
        Error_rate   = ('is_correct', lambda x: (1 - x.mean()) * 100),
        Pos_rate     = ('weak_label', lambda x: x.mean() * 100),
    ).reset_index()
    
    bars = ax.bar(
        type_stats[ans_type_col],
        type_stats['Error_rate'],
        color=['#EF9A9A', '#90CAF9'],
        edgecolor='white', alpha=0.9
    )
    
    for bar, row_t in zip(bars, type_stats.itertuples()):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%\n(N={row_t.N})',
            ha='center', va='bottom', fontsize=10
        )
    
    ax.set_title(f'{ds_name}', fontsize=12)
    ax.set_xlabel('Answer Type')
    ax.set_ylabel('Error Rate (%)')
    ax.set_ylim(0, type_stats['Error_rate'].max() * 1.3 if len(type_stats) > 0 else 100)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/error_rate_by_answer_type.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'💾 Saved → {PLOTS_DIR}/error_rate_by_answer_type.png')

## 💡 Cell 9: Phân Tích Lucky Guess — Ước Lượng Noise Trong Weak Labels

In [ ]:
# Ước tính tỷ lệ lucky guess (False Positive trong weak labels)
# Nguyên lý: Closed-ended (Yes/No) questions dễ đoán đúng ngẫu nhiên (~50%)
# → Với Yes/No, p(lucky guess) ≈ 0.5
# → Với Open-ended, p(lucky guess) ≈ 1/vocab_size << 1

print('='*65)
print('🎲 Lucky Guess Noise Estimation in Weak Labels')
print('='*65)

for ds_name in DATASET_NAMES:
    key = (ds_name, 'reasoning')
    if key not in all_labeled_dfs:
        continue
    
    df = all_labeled_dfs[key]
    
    print(f'\n{ds_name}:')
    
    # Infer closed vs open
    closed_answers = {'yes', 'no', '0', '1', 'true', 'false'}
    is_closed = df['answer_gt'].apply(
        lambda x: str(x).strip().lower() in closed_answers
    )
    
    n_closed  = is_closed.sum()
    n_open    = (~is_closed).sum()
    
    # Ước tính lucky guess rate
    # Giả sử: 50% closed, 1% open có thể bị lucky guess
    # (conservative estimate)
    p_lucky_closed = 0.50
    p_lucky_open   = 0.02  # rất thấp cho free-form
    
    # Trong số good_label (is_correct=1), bao nhiêu là lucky?
    n_good_closed = df[is_closed & (df['weak_label']==1)].shape[0]
    n_good_open   = df[(~is_closed) & (df['weak_label']==1)].shape[0]
    
    # Trong bad (is_correct=0), không ai là lucky
    # Nhưng trong good, một phần có thể là lucky
    # Ước tính số lượng false positives:
    # Chúng ta không biết thực sự, nhưng có thể bound:
    # FP_max ≈ n_good_closed * p_lucky / (p_lucky + p_correct)
    
    print(f'  N closed questions: {n_closed}, N open: {n_open}')
    print(f'  Good-labeled closed: {n_good_closed}, Good-labeled open: {n_good_open}')
    
    if n_closed > 0:
        # Accuracy closed
        acc_closed = df[is_closed]['is_correct'].mean()
        # Estimated lucky guess fraction in "good" closed samples
        # = P(guess correct) / P(correct overall)
        # Simplified: if acc=80%, and P(lucky)=50%, then
        # ~50/80 = 62.5% of correct answers "could" be lucky
        # But this is very loose bound
        if acc_closed > 0:
            lucky_fraction_bound = min(p_lucky_closed / acc_closed, 1.0)
            fp_est = n_good_closed * lucky_fraction_bound
            noise_est = fp_est / len(df) * 100
            print(f'  Estimated FP (lucky guess) — closed only: ~{fp_est:.0f} samples ({noise_est:.1f}% of total)')
    
    print(f'  → Weak label noise is bounded but acceptable for a verifier')
    print(f'  → CLIP verification (Phase 3) provides an INDEPENDENT signal')
    print(f'    to correct lucky-guess false positives')

print('\n💡 Kết luận: Weak labels không hoàn hảo nhưng đủ tốt làm training signal.')
print('   Phase 3 (CLIP) + Phase 4 (Self-Consistency) bổ sung tín hiệu độc lập.')

## 🔀 Cell 10: Thống Kê Cuối Cùng & Kiểm Tra Files Output

In [ ]:
print('='*70)
print('PHASE 2 — FINAL OUTPUT CHECK')
print('='*70)

print(f'\n📁 Files created in: {VERIFIER_DIR}')
total_files = 0
for root, dirs, files in os.walk(VERIFIER_DIR):
    for f in sorted(files):
        full_path = os.path.join(root, f)
        size_kb = os.path.getsize(full_path) / 1024
        print(f'  ✅ {f:55s} ({size_kb:7.1f} KB)')
        total_files += 1

print(f'\nTotal files: {total_files}')

print(f'\n📁 Plots created in: {PLOTS_DIR}')
for root, dirs, files in os.walk(PLOTS_DIR):
    for f in sorted(files):
        full_path = os.path.join(root, f)
        size_kb = os.path.getsize(full_path) / 1024
        print(f'  🖼️  {f:55s} ({size_kb:7.1f} KB)')

print()
print('='*70)
print('PHASE 2 COMPLETE!')
print('='*70)
print()
print('📋 Summary of what was created:')
print('  • Weak labels (good/bad) cho cả 3 datasets × 2 strategies')
print('  • Train/Val/Test splits (60/20/20, stratified) cho Verifier')
print('  • Phân tích imbalance, rationale length, answer type, per-region')
print('  • Lucky guess noise estimation')
print()
print('🚀 Next: Chạy notebook 03_clip_verifier.ipynb')
print('   → Load BiomedCLIP, tính similarity score cho từng sample')
print('   → Phân tích discrimination power: AUC-ROC, Cohen\'s d')